# Qwen2.5 through paged-kv-cache

Downloads Qwen2.5-0.5B-Instruct and answers four chat prompts that share a system prompt. Every layer's K/V lives in paged-kv-cache's block cache: the scheduler decides which blocks each token goes in, paged attention reads them back, and the later prompts reuse the system prompt's blocks from the prefix cache. The same prompts then run through candle's own Qwen2 implementation, and every generated token must match. The last cell repeats the check from Python, through the `paged_kv` package, against Hugging Face `transformers` on PyTorch.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. A CPU runtime works too; it just skips the run with the cache on the GPU.
2. The first run takes several minutes, most of it compiling.

In [ ]:
# Set BRANCH to test a pull request before it is merged.
REPO = "https://github.com/SohamDutta02/paged-kv-cache"
BRANCH = "main"
MAX_NEW_TOKENS = 64

In [ ]:
import glob, os, shutil, subprocess

!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":/usr/local/cuda/bin:" + os.environ["PATH"]

HAS_GPU = shutil.which("nvidia-smi") is not None
if HAS_GPU:
    # cudarc loads NVRTC at runtime. Use the toolkit's copy if there is one, else install it.
    nvrtc = sorted(glob.glob("/usr/local/cuda*/lib64/libnvrtc.so*"))
    if nvrtc:
        lib = os.path.dirname(nvrtc[0])
    else:
        subprocess.run(["pip", "install", "-q", "nvidia-cuda-nvrtc-cu12"], check=True)
        import nvidia.cuda_nvrtc
        lib = os.path.join(nvidia.cuda_nvrtc.__path__[0], "lib")
        if not os.path.exists(os.path.join(lib, "libnvrtc.so")):
            os.symlink("libnvrtc.so.12", os.path.join(lib, "libnvrtc.so"))
    os.environ["LD_LIBRARY_PATH"] = lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("GPU runtime:", HAS_GPU)

In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download(
    "Qwen/Qwen2.5-0.5B-Instruct",
    allow_patterns=["config.json", "tokenizer.json", "model.safetensors"],
)
print(MODEL_DIR)

In [ ]:
!rm -rf paged-kv-cache && git clone -q --depth 1 --branch {BRANCH} {REPO}
!cd paged-kv-cache && git log -1 --format='Testing %h: %s'

In [ ]:
# K/V cache in host memory.
!cd paged-kv-cache && cargo run --release -q -p paged-kv-model --example qwen -- {MODEL_DIR} {MAX_NEW_TOKENS}

In [ ]:
# K/V cache on the GPU, with attention in paged-kv-cuda's kernel.
if HAS_GPU:
    !cd paged-kv-cache && cargo run --release -q -p paged-kv-model --features cuda --example qwen -- {MODEL_DIR} {MAX_NEW_TOKENS}
else:
    print("No GPU on this runtime, so this run is skipped.")

In [ ]:
# The same check from Python: paged_kv against Hugging Face transformers on PyTorch.
!pip install -q ./paged-kv-cache/crates/paged-kv-py
!python paged-kv-cache/crates/paged-kv-py/examples/compare_with_transformers.py {MODEL_DIR} {MAX_NEW_TOKENS}